# Аналіз і обробка часових рядів

In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Завантаження даних

In [4]:
notebook_path = os.path.abspath('Lecture.ipynb')
json_file = os.path.join(os.path.dirname(notebook_path), 'Stalker2.json')
df = None
with open(json_file, 'r') as file:
    json_data = json.loads(file.read())
    df = pd.json_normalize(json_data)

In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df

,player_count,timestamp
0,2127,2025-09-23 10:15:01.557394
1,2198,2025-09-23 11:15:01.297260
2,2863,2025-09-23 12:15:01.868774
3,3457,2025-09-23 13:15:01.901764
4,3972,2025-09-23 14:15:01.963659
...,...,...
2009,2332,2025-12-16 06:15:01.481373
2010,2434,2025-12-16 07:15:01.977042
2011,2863,2025-12-16 08:15:01.861954
2012,3271,2025-12-16 09:15:01.985525


# Попередня обробка (Preprocessing)

In [6]:
df.dropna(inplace=True)
df['player_count'].std()

np.float64(2326.530733911141)

In [ ]:
month_df = df.query('timestamp >= "2025-11-16 00:00:00"')
fig = px.area(
    data_frame=month_df,
    x='timestamp',
    y='player_count',
    title='Кількість гравців у S.T.A.L.K.E.R. 2 за листопад-грудень 2025 р.',
    labels={'timestamp': 'Час', 'player_count': 'Гравці'},
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

## Тест ADF на стаціонарність

In [8]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(month_df['player_count'], autolag='AIC')
print(f'ADF Statistic: {adf_result[0]}\np-value: {adf_result[1]}')

ADF Statistic: -1.627421705616603
p-value: 0.46884058140632967


p-значеня більше 0.05, тому часовий ряд **не є** стаціонарним.

## Декомпозиція на сезонність і циклічність

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposed = seasonal_decompose(month_df['player_count'], model='additive', period=24)
decomposed_df = pd.DataFrame({'Original': decomposed.observed, 'Seasonal': decomposed.seasonal, 'Cyclic': decomposed.trend, 'Residual': decomposed.resid, 'timestamp': month_df['timestamp']})
fig = px.line(
    data_frame=decomposed_df,
    x='timestamp',
    y=['Original', 'Seasonal', 'Cyclic'],
    title='Декомпозиція на сезонність, циклічність і залишки',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

In [ ]:
month_df.insert(1, 'norm', (month_df['player_count'] - month_df['player_count'].min()) / (month_df['player_count'].max() - month_df['player_count'].min()))
month_df.insert(1, 'standard', (month_df['player_count'] - month_df['player_count'].mean()) / month_df['player_count'].std())
fig = px.line(
    data_frame=month_df,
    x='timestamp',
    y=['norm', 'standard'],
    title='Нормалізація і стандартизація',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

## Згладжування

In [ ]:
month_df.insert(1, 'norm_ma', month_df['norm'].rolling(window=5, win_type='gaussian', center=True).mean(std=month_df['norm'].std()))
month_df.insert(1, 'norm_ewm', month_df['norm'].ewm(span=5).mean())
fig = px.line(
    data_frame=month_df,
    x='timestamp',
    y=['norm_ma', 'norm_ewm'],
    title='Згладжування (Moving Average & Exponential)',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

## Поділ на train, validation і testing датасети (60/20/20)

In [12]:
month_df.insert(1, 'shifted_day', month_df['norm_ewm'].shift(periods=24))

train_size = int(len(month_df) * 0.8)
val_size = int(len(month_df) * 0.1)

train = month_df.iloc[:train_size].drop(['norm_ma', 'norm', 'standard', 'player_count'], axis=1)
val = month_df.iloc[train_size:train_size + val_size].drop(['norm_ma', 'norm', 'standard', 'player_count'], axis=1)
test = month_df.iloc[train_size + val_size:].drop(['norm_ma', 'norm', 'standard', 'player_count'], axis=1)

train.reset_index(inplace=True)
val.reset_index(inplace=True)
test.reset_index(inplace=True)

train

,index,shifted_day,norm_ewm,timestamp
0,1283,NaN,0.247776,2025-11-16 00:15:01.303777
1,1284,NaN,0.222883,2025-11-16 01:15:02.105491
2,1285,NaN,0.204997,2025-11-16 02:15:01.493667
3,1286,NaN,0.191899,2025-11-16 03:15:01.391757
4,1287,NaN,0.182409,2025-11-16 04:15:01.801531
...,...,...,...,...
579,1862,0.293326,0.274639,2025-12-10 03:15:01.650330
580,1863,0.265592,0.255128,2025-12-10 04:15:01.238869
581,1864,0.240763,0.237110,2025-12-10 05:15:02.000196
582,1865,0.222446,0.221749,2025-12-10 06:15:01.795292


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=test['timestamp'], y=test['norm_ewm'], mode='lines', name='Test'))
fig.update_layout(
    title='Поділ на train/validation/test',
    xaxis_title='Час',
)
fig.show()

# Аналіз часового ряду

## Автокореляція з різними зсувами

In [ ]:
#month_df.insert(1, 'shifted_day', month_df['norm_ewm'].shift(periods=24))

fig = px.line(
    data_frame=month_df,
    x='timestamp',
    y=['norm_ewm', 'shifted_day'],
    title='Зсув на 24 години',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

In [15]:
month_df['norm_ewm'].corr(month_df['shifted_day'])

np.float64(0.8244822436417409)

In [ ]:
month_df.insert(1, 'shifted_half_day', month_df['norm_ewm'].shift(periods=12))

fig = px.line(
    data_frame=month_df,
    x='timestamp',
    y=['norm_ewm', 'shifted_half_day'],
    title='Зсув на 12 годин',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

In [17]:
month_df['norm_ewm'].corr(month_df['shifted_half_day'])

np.float64(-0.4259216676709549)

In [ ]:
month_df.insert(1, 'shifted_week', month_df['norm_ewm'].shift(periods=24 * 7))

fig = px.line(
    data_frame=month_df,
    x='timestamp',
    y=['norm_ewm', 'shifted_week'],
    title='Зсув на тиждень',
    color_discrete_sequence=px.colors.qualitative.Dark2,
)
fig.show()

In [19]:
month_df['norm_ewm'].corr(month_df['shifted_week'])

np.float64(0.8552235170744682)

## MA, ARMA & ARIMA

In [20]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

def eval(true, pred):
    r2 = r2_score(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    mape = mean_absolute_percentage_error(true, pred) * 100
    print(f'R2: {r2:.2f}\nRMSE: {rmse:.2f}\nMAPE: {mape:.2f}%\n')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model_ma = ARIMA(train['norm_ewm'], order=(0, 0, 1)).fit()
forecast=model_ma.forecast(steps=len(val))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='MA Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою MA',
    xaxis_title='Час',
)
fig.show()

In [22]:
eval(val['norm_ewm'], forecast)

R2: -0.41
RMSE: 0.14
MAPE: 41.42%



In [ ]:
model_arma = ARIMA(train['norm_ewm'], order=(24, 0, 1)).fit()
forecast=model_arma.forecast(steps=len(val))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='ARMA Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою ARMA',
    xaxis_title='Час',
)
fig.show()

In [24]:
eval(val['norm_ewm'], forecast)

R2: 0.78
RMSE: 0.05
MAPE: 15.04%



In [ ]:
model_arima = ARIMA(train['norm_ewm'], order=(24, 1, 1)).fit()
forecast=model_arima.forecast(steps=len(val))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='ARIMA Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою ARIMA',
    xaxis_title='Час',
)
fig.show()

In [26]:
eval(val['norm_ewm'], forecast)

R2: 0.96
RMSE: 0.02
MAPE: 4.57%



## Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

train.dropna(inplace=True)
train.reset_index(inplace=True)
model_rfr = RandomForestRegressor(n_estimators=100).fit(pd.DataFrame(train['norm_ewm']), train['shifted_day'])
forecast = model_rfr.predict(pd.DataFrame(val['norm_ewm']))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='Random Forest Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою Random Forest',
    xaxis_title='Час',
)
fig.show()

In [28]:
eval(val['norm_ewm'], forecast)

R2: 0.83
RMSE: 0.05
MAPE: 9.45%



## Gradient Boost Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model_gbr = GradientBoostingRegressor(n_estimators=100).fit(pd.DataFrame(train['norm_ewm']), train['shifted_day'])
forecast = model_gbr.predict(pd.DataFrame(val['norm_ewm']))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='Gradient Boost Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою Gradient Boost',
    xaxis_title='Час',
)
fig.show()

In [30]:
eval(val['norm_ewm'], forecast)

R2: 0.94
RMSE: 0.03
MAPE: 5.77%



## SVM

In [ ]:
from sklearn.svm import SVR

model_svr = GradientBoostingRegressor(n_estimators=100).fit(pd.DataFrame(train['norm_ewm']), train['shifted_day'])
forecast = model_svr.predict(pd.DataFrame(val['norm_ewm']))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='Support Vector Machine Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою Support Vector Machine',
    xaxis_title='Час',
)
fig.show()

In [57]:
eval(val['norm_ewm'], forecast)

R2: 0.94
RMSE: 0.03
MAPE: 5.77%



## LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout

model_lstm = Sequential()
model_lstm.add(Input(shape=pd.DataFrame(train['norm_ewm']).shape))
model_lstm.add(LSTM(100, activation='relu'))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(pd.DataFrame(train['norm_ewm']), train['shifted_day'], epochs=50, batch_size=32)

forecast = model_lstm.predict(pd.DataFrame(val['norm_ewm']))[:, 0]

fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'], y=train['norm_ewm'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=val['norm_ewm'], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=val['timestamp'], y=forecast, mode='lines', name='LSTM Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою LSTM',
    xaxis_title='Час',
)
fig.show()

In [34]:
eval(val['norm_ewm'], forecast)

R2: 0.90
RMSE: 0.04
MAPE: 11.01%



## TFT - Temporal Fusion Transformer

In [38]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
import torch
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
import random
import tensorflow as tf 
import tensorboard as tb
from sklearn import set_config
from sklearn.preprocessing import StandardScaler

In [ ]:
max_prediction_length = len(val) 
max_encoder_length = len(train)
train.insert(1, 'time_idx', train.index.astype(int))
train.insert(1, 'group_id', '1')
train.drop('level_0', axis=1, inplace=True)

In [43]:
set_config(transform_output="pandas")
training = TimeSeriesDataSet(
    train,
    time_idx="time_idx",
    target="norm_ewm",
    group_ids=["group_id"], 
    min_encoder_length=max_prediction_length,  # keep encoder length long (as it is in the validation set)
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_unknown_reals=[
        "norm_ewm",
    ],
    lags={'norm_ewm': [24, 24 * 7]},
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# create validation set (predict=True) which means to predict the last max_prediction_length points in time
# for each series
validation = TimeSeriesDataSet.from_dataset(training, train, predict=True, stop_randomization=True)

# create dataloaders for model
batch_size = 128  # set this between 32 to 128
num_cpu_cores = os.cpu_count()

print("Starting heavy data pre-processing stage...")

# 1. Create temporary DataLoaders with high num_workers to quickly generate all batches
#    This leverages all your CPU cores to calculate lags/scales in parallel
temp_train_loader = training.to_dataloader(
    train=True, 
    batch_size=batch_size, 
    num_workers=num_cpu_cores
)
temp_val_loader = validation.to_dataloader(
    train=False, 
    batch_size=batch_size, 
    num_workers=num_cpu_cores
)

# 2. Iterate through the temporary loaders and store the resulting batches in a list
#    This executes all the transformation logic once.
train_batches_list = list(temp_train_loader)
val_batches_list = list(temp_val_loader)

print(f"Data pre-processing complete. Generated {len(train_batches_list)} training batches.")

# Now, create simple (fast) DataLoaders that just serve up the pre-processed list
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_batches_list,
    batch_size=None, 
    shuffle=True, 
    num_workers=0
)

val_dataloader = DataLoader(
    val_batches_list,
    batch_size=None, 
    shuffle=False, 
    num_workers=0
)

# train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=num_cpu_cores, pin_memory=False)
# val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=num_cpu_cores, pin_memory=False)

In [ ]:
PATIENCE = 30
MAX_EPOCHS = 120
LEARNING_RATE = 0.03
OPTUNA = False
early_stop_callback = EarlyStopping(monitor="train_loss", min_delta=1e-2, patience=PATIENCE, verbose=False, mode="min")
lr_logger = LearningRateMonitor()  # log the learning rate

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    devices=1, accelerator="cpu",
    enable_model_summary=True,
    gradient_clip_val=0.25,
    #limit_train_batches=10,  # coment in for training, running valiation every 30 batches
    #fast_dev_run=True,  # comment in to check that networkor dataset has no serious bugs
    callbacks=[lr_logger, early_stop_callback],
)


tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    lstm_layers=1,
    hidden_size=8,
    attention_head_size=2,
    dropout=0.2,
    hidden_continuous_size=8,
    output_size=1,  # 7 quantiles by default
    loss=SMAPE(),
    log_interval=10,  # uncomment for learning rate finder and otherwise, e.g. to 10 for logging every 10 batches
    reduce_on_plateau_patience=4
)

tft.to('cpu')
print(f"Number of parameters in network: {tft.size()/1e3:.1f}k")

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

In [75]:
forecast = tft.predict(temp_val_loader, mode="raw", return_x=True)
forecast

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Prediction(output=Output(prediction=tensor([[[0.3272],
         [0.3461],
         [0.3857],
         [0.4336],
         [0.4861],
         [0.5415],
         [0.5907],
         [0.6386],
         [0.6772],
         [0.7047],
         [0.7351],
         [0.7629],
         [0.7774],
         [0.7699],
         [0.7292],
         [0.6579],
         [0.5785],
         [0.5048],
         [0.4426],
         [0.3911],
         [0.3482],
         [0.3110],
         [0.2806],
         [0.2554],
         [0.2405],
         [0.2378],
         [0.2458],
         [0.2593],
         [0.2782],
         [0.3027],
         [0.3275],
         [0.3512],
         [0.3806],
         [0.4137],
         [0.4509],
         [0.4915],
         [0.5293],
         [0.5553],
         [0.5532],
         [0.5180],
         [0.4652],
         [0.4100],
         [0.3622],
         [0.3247],
         [0.2941],
         [0.2678],
         [0.2468],
         [0.2308],
         [0.2231],
         [0.2253],
         [0.23

In [76]:
forecast = forecast.output.prediction[0]
forecast = forecast[:, 0]

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=train['timestamp'][:len(train) - len(forecast)], y=train['norm_ewm'][:len(train) - len(forecast)], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=train['timestamp'][len(train) - len(forecast):], y=train['norm_ewm'][len(train) - len(forecast):], mode='lines', name='Val'))
fig.add_trace(go.Scatter(x=train['timestamp'][len(train) - len(forecast):], y=forecast, mode='lines', name='TFT Prediction'))
fig.update_layout(
    title='Прогнозування з допомогою TFT',
    xaxis_title='Час',
)
fig.show()

In [80]:
eval(train['norm_ewm'][len(train) - len(forecast):], forecast)

R2: 1.00
RMSE: 0.01
MAPE: 0.98%

